**Consigli progetto:**  
L'uso di IA non è vietato, però se la usiamo dobbiamo dichiararlo, conoscere in dettaglio il codice, saper spiegare e motivare le scelte progettuali, verificare che le soluzioni siano corrette e coerenti con quanto studiato. 

Es. se l'applicazione prevede accessi concorrenti a una struttura dati ci si aspetta che questi accessi siano stati testati e si sappia spiegare come vengono gestiti e protetti nel codice. 

Inoltre se si citano articoli su cui si basa la soluzione ovviamente ci si aspetta che lo si abbia letto. 

**Esercizio Chord**:  
Abbiamo $m = 5$ --> GUID da 0 a 31

<img src="imgs/es.png" alt="Esercizio Chord" width="400"/>

1. **Descrivere le DHT e confrontarle con le reti P2P non strutturate:**  
una DHT (Distributed Hash Table) è un'astrazioen distribuita che associa chiavi a valori. Le associazioni (a differenza delle HT locali) sono distribuite tra i peer.  
Peer e risorse hanno GUID nello stesso spazio, e ciò che differenzia i protocolli è la topologia della rete e la metrica di distanza utilizzata tra peer e peer e risorse.  
Rispetto alle reti non strutturate, il vantaggio principale è **ricerca più efficiente**: nei protocolli il lookup costa $O(\log N)$, evitando di dover esplorare una parte ampia della rete usando protocolli di flooding o altri meccanismi simili.  
Lo svantaggio principale è invece **il costo di mantenimento della rete**: nelle reti P2P uno dei problemi principali è il churn --> peer entrano ed escono continuamente o possono guastarsi. In tal caso bisogna aggiornare di conseguenza la rete, le tabelle di routing e le assegnazioni delle chiavi.
2. **Spiegare la finger table del peer 11**:  
Si ricorda la formula: 
    $$FT_p[i] = succ((p + 2^{i-1}) \mod 2^m)$$
    ossia ogni riga contiene il peer responsabile della posizione calcolata, a distanza crescente in modo esponenziale dal peer stesso.  
    Svolgendo quindi i calcoli, per le prime due posizioni della finger table si ottiene il succ del nodo successivo e e di quello ancora dopo --> in questo caso il nodo responsabile di 12 e 13 che è 14.  
    Poi 11 + 2^2 = 15, e il succ di 15 è 18, poi 11 + 2^3 = 19, e il succ di 19 è 22, e 11 + 2^4 = 27, e il succ di 27 è 30 (ricorda che ogni operazione è modulo 32 perché la distanza segue in senso orario il cerchio, se fossi andato oltre 31 si ricomincia da 0).
3. **Cerca la chiave 9 partendo dal peer 11**:  
Il peer 11 gestisce le chiavi 10 e 11 -> non è responsabile di 9 --> chiede al suo successore (14) che a sua volta non è responsabile di 9. Allora secondo il protocollo di routing di Chord sceglierà *il riferimento più avanzato nel percorso circolare verso 9, senza mai superare la chiave*.  
Allora 11 ha nella sua finger table come riferimento più avanzato 28 --> nell'approccio ricorsivo inoltra a lui la richiesta. Allora di nuovo 28 non è responsabile di 9, il suo successore 1 nemmeno --> inoltra a 4, i.e. riferimento più avanzato nella sua finger table precedente a 9.  
Di nuovo ricorsivo: 4 non è responsabile di 9, chiede al successore che è 9 che in questo caso è responsabile della chiave 9 --> fine lookup, si restituisce il peer 9 come responsabile della chiave 9.
4. **Inserire il peer 24 e costruirne la finger table**:  
Le operazioni che segue il peer 24 se deve inserirsi sono:
    1. determina il proprio GUID (24)
    2. esegui la ricerca dell'elemento con GUID pari a 24+1 = 25 per trovare il suo successore (28)
    3. ottiene il suo predecessore chiedendo a 28 chi era il suo di predecessore --> aggiorna puntatori pred e succ ora che li ha ottenuti
    4. fai aggiornare anche i puntatori del succ e pred: il pred di 28 diventa lui (24) e il succ di 22 diventa lui (24)
    5. riceve da 28 le chiavi per il proprio intervallo (quelle comprese tra se stesso e il suo predecessore, i.e. 22, 23 e 24)
    6. costruisce la propria finger table facendo lookup per ogni posizione della tabella, ossia calcolando $succ((24 + 2^{i-1}) \mod 32)$ per i = 1, 2, 3, 4, 5.

**Esercizio metà programma**:
1. Confrontare l’edge computing con il cloud computing, evidenziandone i principali vantaggi e limiti. Presentare un esempio di applicazione distribuita nel compute computing, con l’obiettivo di migliorarne tempo di risposta, disponibilità ed efficienza energetica.
2. Considerando un algoritmo di auto-scaling orizzontale, spiegare perché possono verificarsi oscillazioni nel numero di repliche (continue decisioni di scale-out e scale-in). Fornire un esempio di scenario in cui un algoritmo di scaling predittivo risulta più efficace di uno di scaling reattivo.

**1:**  
L'edge computing colloca le risorse di calcolo e memorizzazione vicino agli utenti e alle sorgenti dei dati --> ciò permette di ridurre latenza di rete, limitare il trasferimento dei dati e mantenere localmente info sensibili, con quindi possibili vantaggi per privacy e consumi energetici. Il limite principale è la minore disponibilità di risorse rispetto al cloud. 

Il cloud computing offre maggiore capacità computazionale e di memorizzazione e infrastrutture di sicurezza più robuste, però la distanza aumenta latenza e il trasferimento dei dati porta a maggiori consumi energetici.

Un esempio di applicazione nel compute continuum è un sistema di videosorveglianza intelligente, dove all'edge si analizzano con un modello leggero i video per rilevare eventi sospetti, e al cloud si inviano solo quelli che richiedono analisi più approfondita.  
In questo modo si migliora il **tempo di risposta** (elaborazione locale nell'edge), **efficienza energetica** (meno dati da trasferire) e **disponibilità** (il sistema può continuare a funzionare anche se la connessione al cloud è interrotta, in questo senso è anche utile replicare i modelli di analisi leggeri su più edge node).

**2:**  
Le oscillazioni nel numero di repliche possono verificarsi se l'algoritmo di autoscaling prende decisioni troppo frequenti o usa soglie configurate male: es. uno scale-out che riduce l'utilizzo medio delle repliche in modo eccessivo/"troppo presto" potrebbe portare subito ad uno scale-in e così via.  
Il fenomeno può essere aggravato per via dei ritardi di avvio delle risorse.

Per limitare le oscillazioni si devono usare soglie opportunamente distanziate e scelte specificatamente per l'applicazione, e si può adottare ad esempuo un **periodo di freezing** dopo ogni decisione di scaling di modo da attendere e misurare l'effetto della decisione prima di prenderne subito un'altra.

Uno scaling predittivo è particolarmente efficace in scenari dove il carico di lavoro segue pattern prevedibili, ad es. sito di e-commerce che riceve picchi di traffico durante le festività o promozioni.  
Qui un algoritmo predittivo può instanziare le risorse in anticipo, a differenza di uno reattivo che reagirebbe solo dopo aver rilevato il sovraccarico --> si evita il tempo di instanziazione delle risorse che altrimenti sarebbe significativo.